# 04 — Consolidated analysis

Everything that compares regions to each other, for **both** lines. Nothing here loads a
region for its own sake: it reads the `metrics.csv` files that `03A` and `03B` write in their
step 3, so a change to one region's thresholds means re-running that one block of `03A`, not
this notebook and not all five regions.

Two sections:

1. **Line A (NOAA)** — the cross-region summary, the 24 h fits, and the amplitude-vs-Wilson
   depression relation. This used to live at the tail of `03A` (cells 88–106), which meant
   analysing a single region also re-ran the conclusions for all of them.
2. **Line B (DS0N)** — amplitude estimators compared across DS00–DS09, and the correlations
   between amplitude, area and field.

### What changed from the previous version

- the hardcoded `/home/thomas/Projects/...` path is gone; paths come from `src.config`
  (the old one does not resolve from this working directory at all)
- the local `plot_time_series` that shadowed the `src` one is gone
- the three different DS lists (`00–09` in one cell, a hand-picked five in another, `00–11`
  on disk) are now just `config.DS0N_IDS`
- the silent `nrows=250` truncation is now an explicit, named window
- `fft_amplitude`, `_prepare`, `sine_amplitude` and `mag_amplitude` moved into
  `src/spectra.py` and `src/oscillation.py`, where `03A` and `03B` can use them too

In [ ]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import numpy as np
import pandas as pd

from src import comparison, config, oscillation, spectra

# Window for the amplitude estimators, in hours from each series' first frame.
# (0, 50) is about two 24 h cycles — the minimum for amplitude, phase and offset to
# separate. None uses the whole series, but then the spot crosses half the disk and the
# mean area stops being comparable between datasets.
WINDOW_H = (0.0, 50.0)
PERIOD_BAND_H = (16.0, 36.0)   # where to look for the "24 h" peak
SAVE = False

print(f'processed NOAA regions : {[p.name for p in config.processed_regions()]}')
print(f'DS0N datasets          : {config.DS0N_IDS}')

---

# Line A — NOAA

## The regions, side by side

Loaded once here and reused by every cell below.

In [ ]:
from src.analysis import compute_metrics
from src.loaders import load_noaa_region

all_metrics = {}
for region_dir in config.processed_regions():
    params = config.params_for(region_dir)
    d = load_noaa_region(region_dir, raw_dir=config.RAW_DIR / region_dir.name,
                         **config.loader_kwargs(params))
    all_metrics[region_dir.name] = (d, compute_metrics(d))
    print(f'{region_dir.name}: {d["n_t"]} frames, {d["time_h"][-1]:.1f} h')

summary = pd.DataFrame(comparison.region_summary_rows(all_metrics))
summary

## Umbral velocity and hot-spot field, all regions on shared axes

The point of overlaying them is the phase: if every region's umbra rises and falls together,
what they share is the observatory, not the Sun.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_v, ax_b) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for name, (d, m) in all_metrics.items():
    ax_v.plot(d['time_h'], m['mean_dop_umb'] - m['mean_dop_quiet'], lw=0.8, label=name)
    if 'mean_mag_hotspot' in m:
        ax_b.plot(d['time_h'], m['mean_mag_hotspot'], lw=0.8, label=name)

ax_v.set_ylabel('umbra − quiet sun  (m/s)')
ax_v.set_title('Umbral velocity relative to the quiet sun')
ax_b.set_ylabel('hot spot mean B  (G)')
ax_b.set_xlabel('Time from the first frame (h)')
for ax in (ax_v, ax_b):
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## The 24 h fit, per region

`fit_diurnal` is a linear least-squares fit of `c + a·sin(2πt/P) + b·cos(2πt/P)` at a
**fixed** period, giving `amplitude = hypot(a, b)` with a real uncertainty.

Two fits per region, over **identical points**:

- **absolute** umbral velocity
- **umbra − quiet sun**

That pair is the instrumental control. Over the same mask the linear fit of `(umbra − quiet)`
is exactly the fit of umbra minus the fit of quiet, so the ratio of the two amplitudes says
how much of the signal is umbral and how much moved the whole box together. At a 24 h period,
"the whole box together" points at residual SDO orbital velocity rather than at the sunspot.

The windows and the Löptien reference table are in `src/config.py`. NOAA 11117 appears twice
because the table does: 2010-10-27 and 2010-10-28 are one continuous passage in a single
cube, so its two rows are the 0–24 h and 24–48 h stretches and give two independent points.

In [ ]:
by_noaa = {config.region_noaa(pathlib.Path(name)): (name, d, m)
           for name, (d, m) in all_metrics.items()}

fit_rows, skipped = [], []
for noaa, date, theta, area_mm2, b_av, z_div, z_press in config.LOPTIEN_TABLE:
    window = config.FIT_WINDOWS.get((noaa, date))
    if noaa not in by_noaa:
        skipped.append(f'NOAA {noaa} {date}: no processed cube')
        continue
    if window is None:
        skipped.append(f'NOAA {noaa} {date}: no entry in config.FIT_WINDOWS')
        continue

    region_name, d, m = by_noaa[noaa]
    time_h = d['time_h']
    v_abs = m['mean_dop_umb']
    v_rel = m['mean_dop_umb'] - m['mean_dop_quiet']
    label = f'NOAA {noaa} {date}'

    # One shared mask for both fits — this is what makes the two amplitude columns a real
    # control rather than two loosely related numbers.
    shared = np.isfinite(v_abs) & np.isfinite(v_rel)

    fit_abs = oscillation.fit_diurnal(time_h, v_abs, window=window,
                                      period_h=config.FIT_PERIOD_H, mask=shared,
                                      label=f'{label} abs')
    fit_rel = oscillation.fit_diurnal(time_h, v_rel, window=window,
                                      period_h=config.FIT_PERIOD_H, mask=shared,
                                      label=f'{label} rel')

    fit_rows.append(dict(label=label, noaa=noaa, date=date, region=region_name,
                         window=window, theta_deg=theta, area_mm2=area_mm2, b_av=b_av,
                         z_div=z_div, z_press=z_press, time_h=time_h,
                         series=v_abs, series_rel=v_rel,
                         fit_abs=fit_abs, fit_rel=fit_rel))

for note in skipped:
    print(f'  skipped  {note}')

print()
header = (f'{"region":22s} {"window":>12s} {"n":>4s} | {"intercept":>10s} '
          f'{"amplitude":>16s} {"t_max":>7s} {"rms":>7s} | {"intercept":>10s} {"amplitude":>16s}')
print(f'{"":22s} {"":>12s} {"":>4s} | {"--- umbra, absolute (m/s) ---":^43s} | '
      f'{"--- umbra - quiet sun (m/s) ---":^29s}')
print(header)
print('-' * len(header))
for row in fit_rows:
    a, r = row['fit_abs'], row['fit_rel']
    t0, t1 = row['window']
    print(f'{row["label"]:22s} {f"{t0:.0f}-{t1:.0f}":>12s} {a["n"]:4d} | '
          f'{a["intercept"]:10.1f} {a["amplitude"]:8.1f} ± {a["sigma_amplitude"]:5.1f} '
          f'{a["t_max"]:7.1f} {a["rms_residual"]:7.1f} | '
          f'{r["intercept"]:10.1f} {r["amplitude"]:8.1f} ± {r["sigma_amplitude"]:5.1f}')

print()
for row in fit_rows:
    a, r = row['fit_abs']['amplitude'], row['fit_rel']['amplitude']
    if np.isfinite(a) and a > 0:
        note = '   <- mostly common to the whole box' if r / a < 0.5 else ''
        print(f'  {row["label"]:22s} umbral fraction of the amplitude: {r / a:.2f}{note}')

Saved next to the cubes, one row per table entry, so the numbers can be re-read without
re-running any of this.

In [ ]:
fit_csv = config.PROCESSED_DIR / 'diurnal_fits.csv'
pd.DataFrame([{
    'label': row['label'], 'noaa': row['noaa'], 'date': row['date'],
    'region': row['region'], 'period_h': config.FIT_PERIOD_H,
    'window_start_h': row['window'][0], 'window_end_h': row['window'][1],
    'n_points': row['fit_abs']['n'],
    'theta_deg': row['theta_deg'], 'area_mm2': row['area_mm2'], 'b_av_G': row['b_av'],
    'z_div_km': row['z_div'], 'z_press_km': row['z_press'],
    **{f'{k}_abs': row['fit_abs'][k] for k in
       ('intercept', 'amplitude', 'sigma_intercept', 'sigma_amplitude', 't_max', 'rms_residual')},
    **{f'{k}_rel': row['fit_rel'][k] for k in
       ('intercept', 'amplitude', 'sigma_intercept', 'sigma_amplitude', 't_max', 'rms_residual')},
} for row in fit_rows]).to_csv(fit_csv, index=False, float_format='%.4f')
print(f'Fits saved → {fit_csv}  ({len(fit_rows)} rows)')

## The fits themselves

Always look at these before reading anything off the scatter below: a fit with a large `rms`
or a window covering barely one period will still return a confident-looking amplitude.

In [ ]:
comparison.plot_diurnal_fits(fit_rows, series_key='fit_rel', save=SAVE,
                             plots_dir=config.PROCESSED_DIR / 'plots')

## Amplitude vs Wilson depression

The scientific payoff of line A: the fitted 24 h umbral Doppler amplitude against the Wilson
depression Löptien et al. measured with Hinode by a completely independent method.

**With six points, Pearson r describes this sample. It is not evidence of a relationship.**

NOAA 13131 is not in the Löptien catalogue — its 800 km is the Romero (2020) value, carried
here so the region can join the same plot. Its θ, area and B_av are unknown.

In [ ]:
plots_dir = config.PROCESSED_DIR / 'plots'
comparison.plot_amplitude_vs_depression(fit_rows, depth_key='z_div', save=SAVE, plots_dir=plots_dir)
comparison.plot_amplitude_vs_depression(fit_rows, depth_key='z_press', save=SAVE, plots_dir=plots_dir)

### Amplitude vs area

Plotting amplitude against **1/area** described the sample better than normalising the
amplitude *by* area, which over-fitted it. Both are here so the difference is visible rather
than remembered.

In [ ]:
comparison.plot_amplitude_vs_area(fit_rows, inverse=False, save=SAVE, plots_dir=plots_dir)
comparison.plot_amplitude_vs_area(fit_rows, inverse=True, save=SAVE, plots_dir=plots_dir)
comparison.plot_amplitude_vs_depression(fit_rows, depth_key='z_div', normalize_area=True,
                                        save=SAVE, plots_dir=plots_dir)

---

# Line B — DS0N

Read from the `metrics.csv` files `03B` writes, so nothing is loaded from a cube here.

In [ ]:
osc_dfs = {}
for ds in config.DS0N_IDS:
    path = config.DS0N_PROCESSED_DIR / ds / 'metrics.csv'
    if path.exists():
        osc_dfs[ds] = pd.read_csv(path)
    else:
        print(f'{ds}: no metrics.csv — run 03B first')

print(f'{len(osc_dfs)} dataset(s) loaded')
for ds, df in osc_dfs.items():
    print(f'  {ds}: {len(df)} rows, {df["time_h"].iloc[-1]:.1f} h')

## Three amplitude estimators, compared

`max − min` is a bad amplitude estimator: it is contaminated by any surviving trend, by gaps,
and by a single outlier. It is kept here only so the newer numbers stay comparable with the
older ones.

- **`peak_to_peak`** — detrend, then half of (max − min)
- **`fft_amplitude`** — Hann window, zero-padded, coherent-gain corrected, peak searched
  inside the 16–36 h band
- **`fit_amplitude`** — a sinusoidal fit at the period the spectrum found, with a real sigma

`n_cycles` is the one to check first: below about 1.5, the window does not constrain an
amplitude no matter which estimator produced it.

In [ ]:
rows = []
for ds, df in osc_dfs.items():
    window = df[(df['time_h'] >= WINDOW_H[0]) & (df['time_h'] <= WINDOW_H[1])]
    time_h = window['time_h'].to_numpy()

    for key, label in [('mean_dop_umb', 'umbra'),
                       ('mean_dop_umb_minus_quiet', 'umbra - quiet')]:
        if key == 'mean_dop_umb_minus_quiet':
            series = (window['mean_dop_umb'] - window['mean_dop_quiet']).to_numpy()
        else:
            series = window[key].to_numpy()

        summary = oscillation.amplitude_summary(time_h, series, band_h=PERIOD_BAND_H,
                                                label=f'{ds} {label}')
        rows.append({'dataset': ds, 'series': label,
                     'area_umb': float(window['area_umb'].mean()), **summary})

osc = pd.DataFrame(rows)
osc.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (var, title) in zip(axes, [('fft_amplitude', 'FFT peak amplitude'),
                                   ('fit_amplitude', 'sinusoidal fit amplitude')]):
    for label, marker in [('umbra', 'o'), ('umbra - quiet', 's')]:
        sub = osc[osc['series'] == label]
        yerr = sub['fit_sigma'] if var == 'fit_amplitude' else None
        ax.errorbar(sub['area_umb'], sub[var], yerr=yerr, fmt=marker, ms=7, lw=0,
                    capsize=3, elinewidth=1, label=label)
        for _, row in sub.iterrows():
            ax.annotate(row['dataset'], (row['area_umb'], row[var]),
                        textcoords='offset points', xytext=(6, 3), fontsize=8, color='0.4')
    finite = osc[osc['series'] == 'umbra']
    ok = np.isfinite(finite['area_umb']) & np.isfinite(finite[var])
    if ok.sum() > 2:
        r = float(np.corrcoef(finite['area_umb'][ok], finite[var][ok])[0, 1])
        ax.set_title(f'{title}   r = {r:+.2f} (n = {int(ok.sum())}, umbra)')
    else:
        ax.set_title(title)
    ax.set_xlabel('mean umbral area (px)')
    ax.set_ylabel('amplitude (m/s)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Correlations

A screening tool on ten datasets: it says which pairs are worth a scatter plot, and nothing
more. Every coefficient rests on as many points as there are rows.

In [ ]:
wide = osc[osc['series'] == 'umbra'].set_index('dataset')[
    ['area_umb', 'fft_amplitude', 'fit_amplitude', 'peak_to_peak', 'fft_period_h']]
comparison.correlation_heatmap(wide, title='DS0N — umbral quantities',
                               save=SAVE, plots_dir=config.DS0N_PROCESSED_DIR / 'plots')

## Spectral peaks across the datasets

`03B` writes a peak table per dataset in step 5. Pooling them shows whether the datasets
agree on which periods exist, or whether each one finds its own.

In [ ]:
peak_frames = []
for ds in config.DS0N_IDS:
    path = config.DS0N_PROCESSED_DIR / ds / 'plots' / 'fft_peaks.csv'
    if path.exists():
        df = pd.read_csv(path)
        # Peak tables written before the spectra merge call the column `rel_amplitude`;
        # `spectra.save_peak_table` now writes `relative`, because the same table also
        # holds relative *power* when the spectrum is a PSD. Accept either, so files
        # already on disk stay readable without re-running 03B.
        if 'relative' not in df.columns and 'rel_amplitude' in df.columns:
            df = df.rename(columns={'rel_amplitude': 'relative'})
        df.insert(0, 'dataset', ds)
        peak_frames.append(df)

if peak_frames:
    peaks = pd.concat(peak_frames, ignore_index=True)
    fig, ax = plt.subplots(figsize=(12, 6))
    for region, colour in [('Umbra', 'green'), ('Penumbra', 'purple'),
                           ('Quiet Sun', 'darkorange')]:
        sub = peaks[peaks['region'] == region]
        ax.scatter(sub['period_min'], sub['relative'], s=28, alpha=0.7,
                   color=colour, label=region)
    ax.set_xscale('log')
    ax.set_xlabel('Period (min)')
    ax.set_ylabel('Relative amplitude')
    ax.set_title(f'FFT peaks across {peaks["dataset"].nunique()} DS0N dataset(s)')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(peaks.head(20).to_string(index=False))
else:
    print('No fft_peaks.csv found — run 03B step 5 with SAVE = True first.')

---

## Where this leaves things

- The umbra and the penumbra move differently from the quiet sun, but the quiet sun's own
  24 h signal is large enough that the *absolute* umbral amplitude is not a measurement of
  the sunspot. The `umbra − quiet` column is the one to quote.
- That the quiet-sun oscillation comes out in antiphase suggests the cleaning removed more
  than it should have. Subtracting the quiet-sun cube from the region cubes directly, rather
  than subtracting per-frame means, would give relative velocities and is the obvious next
  step.
- The magnetogram residual and the Doppler signal share a period. Whether they are genuinely
  in phase, and whether that survives the μ (geometry) check in step 7 of `03A`/`03B`, is
  still open — see `03T_analysis_sandbox.ipynb`.